In [6]:
import torch
from torch import nn
from torchsummary import summary
from dataclasses import dataclass, field

In [3]:
class PreMLPReshape(nn.Module):
    def __init__(self, mlp_input_dim: int):
        super().__init__()
        self.mlp_input_dim = int(mlp_input_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.flatten(x, start_dim=1)  # (B, -1)
        if x.shape[1] != self.mlp_input_dim:
            raise RuntimeError(
                f"PreMLPReshape: got {x.shape[1]} features; expected {self.mlp_input_dim}."
            )
        return x


In [5]:
@dataclass
class MLPConfig:
    input: int = 109
    output: int = 10
    neurons: list[int] = field(default_factory=[1, 2, 3])
    layers: int = 3

class NeuralNetwork(nn.Module):
  def __init__(self, cfg: MLPConfig):

      super().__init__()
      self.input= cfg.input
      self.output= cfg.output
      self.layers= cfg.layers
      self.neurons=cfg.neurons
      assert len(self.neurons)==self.layers, "Número de capas y neuronas no coinciden"

      self.linnears = []

      for i, j in enumerate(self.neurons):
        if i == 0:
          self.linnears.append(nn.Linear(self.input,j))
          self.linnears.append(nn.ReLU())
        elif i <= self.layers:
          self.linnears.append(nn.Linear(self.neurons[i-1],j))
          self.linnears.append(nn.ReLU())

      self.linnears.append(nn.Linear(j,self.output))
      print(self.linnears)


  def forward(self, x):
      for layer in self.linnears:
          x = layer(x)

      print(x)
      return x

In [4]:
class PostMLPReshape(nn.Module):
    def __init__(self, *target_shape: int):
        super().__init__()
        if len(target_shape) == 1 and isinstance(target_shape[0], (list, tuple)):
            target_shape = tuple(target_shape[0])
        self.target_shape = tuple(int(d) for d in target_shape)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B = x.shape[0]
        F = x.shape[1]

        prod = 1
        for d in self.target_shape:
            prod *= d
        if F != prod:
            raise RuntimeError(
                f"PostMLPReshape: no puedo convertir de (B, {F}) a (B, {self.target_shape}); "
                f"se requieren {prod} features."
            )
        return x.reshape(B, *self.target_shape)


In [7]:
@dataclass
class Conv2DHeadConfig:
    in_channels: int          # = C que sale del PostMLPReshape
    out_channels: int = 16
    kernel_size: int = 3
    stride: int = 1
    padding: str | int = "same"   # "same" => padding automático
    use_bn: bool = True
    activation: str | None = "relu"  # "relu", "gelu", "silu" o None

class Conv2DHead(nn.Module):
    def __init__(self, cfg: Conv2DHeadConfig):
        super().__init__()
        if cfg.padding == "same":
            # para stride=1 y dilatación=1: SAME -> k//2
            pad = cfg.kernel_size // 2
        else:
            pad = int(cfg.padding)

        layers = []
        layers.append(
            nn.Conv2d(
                in_channels=cfg.in_channels,
                out_channels=cfg.out_channels,
                kernel_size=cfg.kernel_size,
                stride=cfg.stride,
                padding=pad,
                bias=not cfg.use_bn,
            )
        )
        if cfg.use_bn:
            layers.append(nn.BatchNorm2d(cfg.out_channels))

        if cfg.activation:
            act = cfg.activation.lower()
            if   act == "relu": layers.append(nn.ReLU(inplace=True))
            elif act == "gelu": layers.append(nn.GELU())
            elif act == "silu": layers.append(nn.SiLU(inplace=True))
            else: raise ValueError(f"Activación no soportada: {cfg.activation}")

        self.net = nn.Sequential(*layers)
        self.in_channels = cfg.in_channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 4:
            raise RuntimeError(f"Conv2DHead: se esperaba (B, C, H, W) y llegó {tuple(x.shape)}.")
        if x.shape[1] != self.in_channels:
            raise RuntimeError(
                f"Conv2DHead: C={x.shape[1]} no coincide con in_channels={self.in_channels}."
            )
        return self.net(x)

In [10]:
@dataclass
class Conv2DTailConfig:
    in_channels: int          # Debe coincidir con la salida del Conv2DHead
    out_channels: int = 1     # Canales de la reconstrucción final (p.ej., 1 para grises, 3 para RGB)
    kernel_size: int = 3
    stride: int = 1
    padding: str | int = "same"     # "same" -> conserva HxW
    use_bn: bool = False            # En la capa final normalmente se omite BN
    output_activation: str | None = None
    # opciones: None|"sigmoid"|"tanh"|"relu"|"identity" (identity ≡ None)

class Conv2DTail(nn.Module):
    def __init__(self, cfg: Conv2DTailConfig):
        super().__init__()
        if cfg.padding == "same":
            pad = cfg.kernel_size // 2
        else:
            pad = int(cfg.padding)

        layers = [
            nn.Conv2d(
                in_channels=cfg.in_channels,
                out_channels=cfg.out_channels,
                kernel_size=cfg.kernel_size,
                stride=cfg.stride,
                padding=pad,
                bias=not cfg.use_bn,
            )
        ]
        if cfg.use_bn:
            layers.append(nn.BatchNorm2d(cfg.out_channels))

        # Activación de salida (opcional)
        act = (cfg.output_activation or "").lower()
        if act in ("", "none", "identity"):
            pass
        elif act == "sigmoid":
            layers.append(nn.Sigmoid())
        elif act == "tanh":
            layers.append(nn.Tanh())
        elif act == "relu":
            layers.append(nn.ReLU(inplace=True))
        else:
            raise ValueError(f"Activación de salida no soportada: {cfg.output_activation}")

        self.net = nn.Sequential(*layers)
        self.in_channels = cfg.in_channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 4:
            raise RuntimeError(f"Conv2DTail: se esperaba (B, C, H, W) y llegó {tuple(x.shape)}.")
        if x.shape[1] != self.in_channels:
            raise RuntimeError(
                f"Conv2DTail: C={x.shape[1]} no coincide con in_channels={self.in_channels}."
            )
        return self.net(x)

# Integración

Gepeto se volvió loco y lo unió todo en el siguiente código.

In [12]:
# main.py
import torch
from torch import nn
from dataclasses import dataclass, field

# ---------- Configs ----------
@dataclass
class MLPConfig:
    input: int = 109
    output: int = 28*28  # p.ej. queremos re-formar a (1, 28, 28)
    neurons: list[int] = field(default_factory=lambda: [128, 64, 64])  # FIX: lambda
    layers: int = 3

# ---------- Blocks ----------
class PreMLPReshape(nn.Module):
    def __init__(self, mlp_input_dim: int):
        super().__init__()
        self.mlp_input_dim = int(mlp_input_dim)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.flatten(x, start_dim=1)
        if x.shape[1] != self.mlp_input_dim:
            raise RuntimeError(f"PreMLPReshape: got {x.shape[1]} != {self.mlp_input_dim}")
        return x

class NeuralNetwork(nn.Module):
    def __init__(self, cfg: MLPConfig):
        super().__init__()
        assert len(cfg.neurons) == cfg.layers, "Capas y neuronas no coinciden"
        layers: list[nn.Module] = []
        in_f = cfg.input
        for h in cfg.neurons:
            layers += [nn.Linear(in_f, h), nn.ReLU()]
            in_f = h
        layers += [nn.Linear(in_f, cfg.output)]
        self.net = nn.Sequential(*layers)  # FIX: registrar parámetros
    def forward(self, x): return self.net(x)

class PostMLPReshape(nn.Module):
    def __init__(self, *target_shape: int):
        super().__init__()
        if len(target_shape) == 1 and isinstance(target_shape[0], (list, tuple)):
            target_shape = tuple(target_shape[0])
        self.target_shape = tuple(int(d) for d in target_shape)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, F = x.shape[0], x.shape[1]
        prod = 1
        for d in self.target_shape: prod *= d
        if F != prod:
            raise RuntimeError(f"PostMLPReshape: {F} != prod{self.target_shape}={prod}")
        return x.reshape(B, *self.target_shape)

from dataclasses import dataclass
@dataclass
class Conv2DHeadConfig:
    in_channels: int
    out_channels: int = 8
    kernel_size: int = 3
    stride: int = 1
    padding: str | int = "same"
    use_bn: bool = True
    activation: str | None = "relu"

class Conv2DHead(nn.Module):
    def __init__(self, cfg: Conv2DHeadConfig):
        super().__init__()
        pad = cfg.kernel_size // 2 if cfg.padding == "same" else int(cfg.padding)
        layers = [
            nn.Conv2d(cfg.in_channels, cfg.out_channels, cfg.kernel_size, cfg.stride, pad, bias=not cfg.use_bn)
        ]
        if cfg.use_bn: layers.append(nn.BatchNorm2d(cfg.out_channels))
        if cfg.activation:
            a = cfg.activation.lower()
            layers.append({"relu": nn.ReLU(inplace=True), "gelu": nn.GELU(), "silu": nn.SiLU(inplace=True)}[a])
        self.net = nn.Sequential(*layers)
        self.in_channels = cfg.in_channels
    def forward(self, x):
        if x.ndim != 4 or x.shape[1] != self.in_channels:
            raise RuntimeError(f"Conv2DHead: esperado (B,{self.in_channels},H,W), got {tuple(x.shape)}")
        return self.net(x)

@dataclass
class Conv2DTailConfig:
    in_channels: int
    out_channels: int = 1
    kernel_size: int = 3
    stride: int = 1
    padding: str | int = "same"
    use_bn: bool = False
    output_activation: str | None = None

class Conv2DTail(nn.Module):
    def __init__(self, cfg: Conv2DTailConfig):
        super().__init__()
        pad = cfg.kernel_size // 2 if cfg.padding == "same" else int(cfg.padding)
        layers = [nn.Conv2d(cfg.in_channels, cfg.out_channels, cfg.kernel_size, cfg.stride, pad, bias=not cfg.use_bn)]
        if cfg.use_bn: layers.append(nn.BatchNorm2d(cfg.out_channels))
        act = (cfg.output_activation or "").lower()
        if act == "sigmoid": layers.append(nn.Sigmoid())
        elif act == "tanh": layers.append(nn.Tanh())
        elif act == "relu": layers.append(nn.ReLU(inplace=True))
        self.net = nn.Sequential(*layers)
        self.in_channels = cfg.in_channels
    def forward(self, x):
        if x.ndim != 4 or x.shape[1] != self.in_channels:
            raise RuntimeError(f"Conv2DTail: esperado (B,{self.in_channels},H,W), got {tuple(x.shape)}")
        return self.net(x)

# ---------- Reconstruction Block wrapper (opcional) ----------
class ReconstructionBlock(nn.Module):
    def __init__(self, mlp_cfg: MLPConfig, post_shape: tuple[int, int, int],
                 head_out_ch: int = 8, tail_out_ch: int = 1, tail_act: str | None = "sigmoid"):
        super().__init__()
        C, H, W = post_shape
        assert mlp_cfg.output == C*H*W, "cfg.output debe coincidir con C*H*W"
        self.pre = PreMLPReshape(mlp_cfg.input)
        self.mlp = NeuralNetwork(mlp_cfg)
        self.post = PostMLPReshape(C, H, W)
        self.head = Conv2DHead(Conv2DHeadConfig(in_channels=C, out_channels=head_out_ch))
        self.tail = Conv2DTail(Conv2DTailConfig(in_channels=head_out_ch, out_channels=tail_out_ch,
                                                output_activation=tail_act))
    def forward(self, x):
        x = self.pre(x)     # (B, F)
        x = self.mlp(x)     # (B, cfg.output)
        x = self.post(x)    # (B, C, H, W)
        x = self.head(x)    # (B, C', H, W)
        x = self.tail(x)    # (B, Cout, H, W)
        return x

# ---------- Demo ----------
if __name__ == "__main__":
    mlp_cfg = MLPConfig(input=109, output=28*28, neurons=[128, 64, 64], layers=3)
    block = ReconstructionBlock(mlp_cfg, post_shape=(1, 28, 28), head_out_ch=8, tail_out_ch=1, tail_act="sigmoid")
    x = torch.randn(4, 109)           # o (4, 1, 109) etc. — se aplana igual
    y = block(x)
    print("Salida:", y.shape)         # -> (4, 1, 28, 28)


Salida: torch.Size([4, 1, 28, 28])


# Reconstruction Block (PreMLPReshape → MLP → PostMLPReshape → Conv2DHead → Conv2DTail)

Este README documenta **paso a paso** un bloque de reconstrucción en PyTorch compuesto por:
1) `PreMLPReshape` (aplana y valida),
2) `NeuralNetwork` (MLP),
3) `PostMLPReshape` (reconvierte a `C×H×W`),
4) `Conv2DHead` (conv intermedia),
5) `Conv2DTail` (conv final para la salida de reconstrucción).

---

## 0) Propósito

Transformar una entrada arbitraria (vector o tensor) en una **salida 2D** (imagen/mapa) a través de un MLP y un par de convoluciones 2D, lista para usarse con pérdidas como *reconstruction loss* o *causal/casual loss*.

---

## 1) Requisitos

- Python ≥ 3.10  
- PyTorch ≥ 2.0

Instalación:
```bash
pip install torch
```

## 2) Estructura del proyecto sugerida

your_project/

├─ main.py

├─ blocks/

│  ├─ mlp.py               # MLPConfig, NeuralNetwork

│  ├─ reshape.py           # PreMLPReshape, PostMLPReshape

│  ├─ conv.py              # Conv2DHead, Conv2DTail (+ configs)

│  └─ reconstruction.py    # ReconstructionBlock (wrapper)

└─ README.md

## 3) Arquitectura (visión general)

Entrada (B, ...)

   │

   ▼

PreMLPReshape          # (B, ...) → (B, F_in) y valida

   │

   ▼

MLP                    # (B, F_in) → (B, F_out)

   │

   ▼

PostMLPReshape         # (B, F_out) → (B, C, H, W)

   │

   ▼

Conv2DHead             # (B, C, H, W) → (B, C', H, W)

   │

   ▼

Conv2DTail             # (B, C', H, W) → (B, C_out, H, W)

| Debe cumplirse `F_out = C * H * W`.


## 4) Implementación (clases y parámetros)
### 4.1 `MLPConfig` (configuración del MLP)
```python
from dataclasses import dataclass, field

@dataclass
class MLPConfig:
    input: int = 109             # F_in: features esperadas tras el aplanado
    output: int = 28*28          # F_out: debe ser C*H*W para el PostMLPReshape
    neurons: list[int] = field(default_factory=lambda: [128, 64, 64])
    layers: int = 3              # Debe coincidir con len(neurons)
```

Campos:

* `input`: número de features a la entrada del MLP (coincide con el aplanado).

* `output`: features a la salida del MLP; debe cumplir `output = C*H*W`.

* `neurons`: dimensiones de cada capa oculta.

* `layers`: cantidad de capas ocultas; requiere `layers == len(neurons)`.

### 4.2 `NeuralNetwork` (MLP)

```python
import torch
from torch import nn

class NeuralNetwork(nn.Module):
    def __init__(self, cfg: MLPConfig):
        super().__init__()
        assert len(cfg.neurons) == cfg.layers, "Capas y neuronas no coinciden"
        layers: list[nn.Module] = []
        in_f = cfg.input
        for h in cfg.neurons:
            layers += [nn.Linear(in_f, h), nn.ReLU()]
            in_f = h
        layers += [nn.Linear(in_f, cfg.output)]
        self.net = nn.Sequential(*layers)  # registra parámetros

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)
```

Entradas/Salidas:

* Entrada esperada: `(B, cfg.input)`.

* Salida: `(B, cfg.output)`.

### 4.3 `PreMLPReshape` (aplanado y validación)

```python
class PreMLPReshape(nn.Module):
    """
    Aplana cualquier tensor a (B, F) y valida que F == mlp_input_dim.
    """
    def __init__(self, mlp_input_dim: int):
        super().__init__()
        self.mlp_input_dim = int(mlp_input_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.flatten(x, start_dim=1)  # → (B, -1)
        if x.shape[1] != self.mlp_input_dim:
            raise RuntimeError(
                f"PreMLPReshape: got {x.shape[1]} features; expected {self.mlp_input_dim}."
            )
        return x
```

Parámetros:

* `mlp_input_dim`: normalmente `cfg.input`.

* Error típico: “got F != mlp_input_dim” → ajusta la forma de entrada o `cfg.input`.

### 4.4 PostMLPReshape (reformatea a C×H×W)

```python
class PostMLPReshape(nn.Module):
    """
    Convierte (B, F) en (B, C, H, W) verificando F == C*H*W.
    """
    def __init__(self, *target_shape: int):
        super().__init__()
        if len(target_shape) == 1 and isinstance(target_shape[0], (list, tuple)):
            target_shape = tuple(target_shape[0])
        self.target_shape = tuple(int(d) for d in target_shape)  # (C,H,W)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, F = x.shape[0], x.shape[1]
        prod = 1
        for d in self.target_shape:
            prod *= d
        if F != prod:
            raise RuntimeError(
                f"PostMLPReshape: {F} != prod{self.target_shape}={prod}"
            )
        return x.reshape(B, *self.target_shape)
```

Parámetros:

* `target_shape`: tu forma deseada `(C, H, W)`.

Requisito: `cfg.output == C*H*W`.

### 4.5 `Conv2DHeadConfig` y `Conv2DHead` (conv intermedia)

```python
from dataclasses import dataclass

@dataclass
class Conv2DHeadConfig:
    in_channels: int          # = C (salida del PostMLPReshape)
    out_channels: int = 8
    kernel_size: int = 3
    stride: int = 1
    padding: str | int = "same"   # "same" ≈ k//2 para stride=1
    use_bn: bool = True
    activation: str | None = "relu"  # "relu"|"gelu"|"silu"|None

class Conv2DHead(nn.Module):
    """
    Bloque conv2d para aplicar inmediatamente después del PostMLPReshape.
    Entrada esperada: (B, C, H, W).
    """
    def __init__(self, cfg: Conv2DHeadConfig):
        super().__init__()
        pad = cfg.kernel_size // 2 if cfg.padding == "same" else int(cfg.padding)

        layers = [
            nn.Conv2d(cfg.in_channels, cfg.out_channels,
                      kernel_size=cfg.kernel_size,
                      stride=cfg.stride,
                      padding=pad,
                      bias=not cfg.use_bn)
        ]
        if cfg.use_bn:
            layers.append(nn.BatchNorm2d(cfg.out_channels))

        if cfg.activation:
            act = cfg.activation.lower()
            if   act == "relu": layers.append(nn.ReLU(inplace=True))
            elif act == "gelu": layers.append(nn.GELU())
            elif act == "silu": layers.append(nn.SiLU(inplace=True))
            else: raise ValueError(f"Activación no soportada: {cfg.activation}")

        self.net = nn.Sequential(*layers)
        self.in_channels = cfg.in_channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 4:
            raise RuntimeError(f"Conv2DHead: esperado (B, C, H, W); got {tuple(x.shape)}.")
        if x.shape[1] != self.in_channels:
            raise RuntimeError(
                f"Conv2DHead: C={x.shape[1]} != in_channels={self.in_channels}."
            )
        return self.net(x)
```

Campos clave:

* `in_channels`: igual a `C` del `PostMLPReshape`.

* `out_channels`: canales intermedios (`C`).

* `activation`: `relu`/`gelu`/`silu` o `None`.

### 4.6 `Conv2DTailConfig` y `Conv2DTail` (conv final)

```python
@dataclass
class Conv2DTailConfig:
    in_channels: int          # = out_channels del Conv2DHead
    out_channels: int = 1     # canales de la salida final (1 gris, 3 RGB, etc.)
    kernel_size: int = 3
    stride: int = 1
    padding: str | int = "same"
    use_bn: bool = False
    output_activation: str | None = None  # None|"sigmoid"|"tanh"|"relu"

class Conv2DTail(nn.Module):
    """
    Última conv2d del bloque de reconstrucción.
    """
    def __init__(self, cfg: Conv2DTailConfig):
        super().__init__()
        pad = cfg.kernel_size // 2 if cfg.padding == "same" else int(cfg.padding)

        layers = [
            nn.Conv2d(cfg.in_channels, cfg.out_channels,
                      kernel_size=cfg.kernel_size,
                      stride=cfg.stride,
                      padding=pad,
                      bias=not cfg.use_bn)
        ]
        if cfg.use_bn:
            layers.append(nn.BatchNorm2d(cfg.out_channels))

        act = (cfg.output_activation or "").lower()
        if   act in ("", "none", "identity"): pass
        elif act == "sigmoid": layers.append(nn.Sigmoid())
        elif act == "tanh":    layers.append(nn.Tanh())
        elif act == "relu":    layers.append(nn.ReLU(inplace=True))
        else: raise ValueError(f"Activación de salida no soportada: {cfg.output_activation}")

        self.net = nn.Sequential(*layers)
        self.in_channels = cfg.in_channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 4:
            raise RuntimeError(f"Conv2DTail: esperado (B, C, H, W); got {tuple(x.shape)}.")
        if x.shape[1] != self.in_channels:
            raise RuntimeError(
                f"Conv2DTail: C={x.shape[1]} != in_channels={self.in_channels}."
            )
        return self.net(x)
```

Campos clave:

* `out_channels`: canales de salida final.

* `output_activation`: decide el rango de salida (`None`, `sigmoid`, `tanh`, `relu`).

## 5) Wrapper: `ReconstructionBlock`

```python
class ReconstructionBlock(nn.Module):
    """
    PreMLPReshape → MLP → PostMLPReshape → Conv2DHead → Conv2DTail
    """
    def __init__(self, mlp_cfg: MLPConfig, post_shape: tuple[int, int, int],
                 head_out_ch: int = 8,
                 tail_out_ch: int = 1,
                 tail_act: str | None = "sigmoid"):
        super().__init__()
        C, H, W = post_shape
        assert mlp_cfg.output == C*H*W, "cfg.output debe ser igual a C*H*W"

        self.pre  = PreMLPReshape(mlp_cfg.input)
        self.mlp  = NeuralNetwork(mlp_cfg)
        self.post = PostMLPReshape(C, H, W)
        self.head = Conv2DHead(Conv2DHeadConfig(in_channels=C, out_channels=head_out_ch))
        self.tail = Conv2DTail(Conv2DTailConfig(in_channels=head_out_ch,
                                                out_channels=tail_out_ch,
                                                output_activation=tail_act))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pre(x)   # (B, cfg.input)
        x = self.mlp(x)   # (B, cfg.output)
        x = self.post(x)  # (B, C, H, W)
        x = self.head(x)  # (B, head_out_ch, H, W)
        x = self.tail(x)  # (B, tail_out_ch, H, W)
        return x
```
Contratos de forma:

* Pre: `(B, ...) → (B, cfg.input)`

* MLP: `(B, cfg.input) → (B, cfg.output)`

* Post: `(B, cfg.output) → (B, C, H, W)` con `cfg.output = C*H*W`

* Head: `(B, C, H, W) → (B, head_out_ch, H, W)`

* Tail: `(B, head_out_ch, H, W) → (B, tail_out_ch, H, W)`

## 6) Ejemplo mínimo (runnable)

```python
import torch

# 1) Config MLP para que su salida coincida con (C,H,W)
mlp_cfg = MLPConfig(input=109, output=28*28, neurons=[128, 64, 64], layers=3)

# 2) Crear el bloque completo: reconstrucción (1,28,28)
block = ReconstructionBlock(
    mlp_cfg=mlp_cfg,
    post_shape=(1, 28, 28),
    head_out_ch=8,
    tail_out_ch=1,
    tail_act="sigmoid"  # si tus targets están en [0,1]
)

# 3) Entrada de prueba (cualquier forma que aplane a 109)
x = torch.randn(4, 109)  # también (4, 1, 109)
y = block(x)
print("Salida:", y.shape)  # torch.Size([4, 1, 28, 28])
```

## 7) Guía de pérdidas (losses)

* Reconstrucción (valores reales):

  * `output_activation=None`

  * `nn.MSELoss()` o `nn.L1Loss()`

* Binario por píxel con logits:

  * `output_activation=None`

  * `nn.BCEWithLogitsLoss()` (aplica `sigmoid` internamente)

* Probabilidades en [0,1]:

  * `output_activation="sigmoid"`

  * `nn.BCELoss()`

* Salida en [-1,1]:

  * `output_activation="tanh"` y escala tus targets a [-1,1]

## 8) Entrenamiento (snippet)

```python
import torch
from torch import nn, optim

mlp_cfg = MLPConfig(input=109, output=28*28, neurons=[128, 64, 64], layers=3)
block = ReconstructionBlock(mlp_cfg, (1, 28, 28), head_out_ch=8, tail_out_ch=1, tail_act="sigmoid")

criterion = nn.MSELoss()
optimizer = optim.Adam(block.parameters(), lr=1e-3)

for epoch in range(5):
    block.train()
    x = torch.randn(32, 109)
    target = torch.rand(32, 1, 28, 28)  # en [0,1]
    y_hat = block(x)

    loss = criterion(y_hat, target)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch} | Loss {loss.item():.4f}")
```

## 9) Validaciones y errores comunes

1) PreMLPReshape:

* Error: “got F != mlp_input_dim” → ajusta forma de entrada o `cfg.input`.

2) PostMLPReshape:

* Error: “F != prod(target_shape)” → asegura `cfg.output == C*H*W`.

3) Conv2DHead/Tail:

* Error: forma esperada `(B, C, H, W)` o canales no coinciden → revisa `in_channels`.

4) Padding "same":

* Usamos `k//2` (válido con `stride=1`, `dilatación=1`).

* Con `stride≠1`, usa `padding="same"` nativo (si tu PyTorch lo soporta) o calcula padding manualmente.

5) Registro de capas MLP:

* Usa `nn.Sequential`/`nn.ModuleList`. Listas Python no registran parámetros.

## 10) Extensión y personalización

* Cambiar activación del MLP (`ReLU` → `GELU`/`SiLU`).

* Insertar más capas conv entre `Head` y `Tail`.

* Evitar BatchNorm en la salida final si afecta la distribución esperada.

* Cambiar `post_shape=(C,H,W)` y ajustar `cfg.output=C*H*W`.

## 11) FAQ

* ¿Puedo pasar `(B, 1, 109)`?
  Sí, `PreMLPReshape` aplana a `(B, 109)` y valida contra `cfg.input`.

* ¿Puedo usar `padding="same"` directo?
  Sí, si tu versión de PyTorch lo soporta. Aquí se usa `k//2` por compatibilidad.

## 12) Integración con pérdidas externas

```python
# recon_loss.py
import torch.nn as nn

def reconstruction_loss(y_hat, y_true, kind="mse"):
    if kind == "mse":
        return nn.MSELoss()(y_hat, y_true)
    elif kind == "l1":
        return nn.L1Loss()(y_hat, y_true)
    else:
        raise ValueError(f"Unsupported kind: {kind}")

# causal_loss.py (ejemplo simple)
def causal_loss(y_hat, y_true, mask=None):
    diff = (y_hat - y_true) if mask is None else (y_hat - y_true) * mask
    return (diff**2).mean()
```

## 13) Buenas prácticas

* Asegura que targets estén escalados acorde a la activación de salida del `Tail`.

* Mantén `cfg.output = C*H*W` para evitar errores de forma.

* Comienza con `stride=1` y `padding=“same”` para conservar tamaños espaciales.

* Activa `BatchNorm` solo donde ayude a la estabilidad (usualmente en `Head`, no en `Tail`).

## 14) Licencia

* MIT